# Constraint Expressions: subconjunto del lado del servidor

## Motivación

Muchos servidores OPeNDAP contienen colecciones de datasets que, en agregación, describen un dataset completo (por ejemplo, una simulación multianual). En algunos casos, cada dataset puede contener $\sim O(10-100)$ variables; esto es particularmente cierto para datos Nivel 2, consulta por ejemplo [este dataset ATLAS03](http://test.opendap.org:8080/opendap/atlas03/ATL03_20200131234815_05540602_003_01.h5.dmr.html) de NASA. Esto significa que PyDAP debe analizar cientos de variables por archivo. PyDAP es un parser rápido de *DMRs* (la respuesta de [metadatos DAP4](https://opendap.github.io/dap4-specification/DAP4.html#_dmr_declarations)), pero:

1. PyDAP no agrega (todavía) datasets ni URLs.
2. PyDAP no hace comprobaciones CF ni define operaciones basadas en etiquetas.

`xarray`, por otro lado, permite lectura paralela y agregación de múltiples datasets. Sin embargo, a pesar de las excelentes capacidades paralelas de `xarray`, puede ser lento durante la agregación de múltiples datasets porque realiza varias comprobaciones sobre los metadatos para que la colección de archivos sea "segura" de abrir y operar, de acuerdo con la lógica interna de `xarray`. Esto significa que xarray realiza comprobaciones extra para asegurar operaciones seguras basadas en etiquetas, y el tiempo puede crecer con el número de variables.

```{note}
Aunque `xarray` tiene un método para descartar variables del `xarray.Dataset` virtual, solo está disponible **después** de crear el dataset. Es decir, si tienes cientos de archivos, cada uno con cientos de variables, pero solo te interesan 2-3 variables por archivo, el cliente debe analizar y comprobar todas las variables en todos los archivos para crear un dataset del cual luego descartarás todas menos las 2-3 variables de interés.
```

Estos factores, en combinación, representan un obstáculo para la exploración inicial de colecciones de datasets OPeNDAP que contienen muchas `variables`, `Groups` y otros tipos complejos.


## Enfoque

Se pueden agregar `Constraint Expresions` (`CEs`) a la URL del dataset y enviarlas al servidor OPeNDAP remoto para:

1. Solicitar un subconjunto de variables, y
2. Solicitar un subconjunto espacial de variables.


Estas `CEs` son increíblemente potentes desde la perspectiva de la persona usuaria porque `pydap`, como motor backend de `xarray`, puede recibir una respuesta DAP mucho más pequeña desde el servidor OPeNDAP remoto. Es decir, las `CEs informan al servidor OPeNDAP remoto que debe subdividir cerca de los datos`. La respuesta puede ser aproximadamente $\sim O(1)$ segundo más rápida en total, pero al considerar cientos de URLs, el resultado puede tener un impacto significativo en el rendimiento y la experiencia de uso.


A continuación revisamos `Constraint Expressions` con datos reales en servidores OPeNDAP. Como hay dos protocolos DAP distintos (consulta [Pydap como cliente](client)), revisaremos los dos casos por separado.


In [ ]:
from pydap.client import open_url
import numpy as np
import matplotlib.pyplot as plt

## Constraint Expressions en DAP4

Aquí demostramos el uso de CE en arreglos de 1 y 2 dimensiones, en dos datasets distintos:

1. [ATLAS03](http://test.opendap.org:8080/opendap/atlas03/ATL03_20200131234815_05540602_003_01.h5.dmr.html), datos Nivel 2.
2. [Daily MUR Sea Surface Temperature](http://test.opendap.org:8080/opendap/ghrsst/20210102090000-JPL-L4_GHRSST-SSTfnd-MUR-GLOB-v02.0-fv04.1.h5.dmr), datos Nivel 4 proporcionados por PODAAC en JPL.


```{note}
Para ejemplos más interesantes, mira el [ejemplo PACE](notebooks/PACE).
```

### ATLAS03

Este dataset tiene muchos Groups anidados, con muchas variables dentro. Supongamos que solo nos interesan las variables:

1. `delta_time`.
2. `lat_ph`.
3. `lon_ph`.

Las tres variables están dentro del Group `heights`, que a su vez está anidado en el `Group` llamado `gt3r`.

Abramos el archivo remoto de forma ingenua, solicitando todas las variables al servidor OPeNDAP ([Hyrax](https://www.opendap.org/software/hyrax-data-server/) en este caso).


In [ ]:
data_url = 'http://test.opendap.org:8080/opendap/atlas03/ATL03_20200131234815_05540602_003_01.h5'

In [ ]:
%%time
ds = open_url(data_url, protocol='dap4')

Abajo imprimimos todo el árbol de directorios dentro del dataset HDF5.


In [ ]:
ds.tree()

Debemos notar que los mismos nombres de variables aparecen en otros grupos a lo largo del archivo, en otros `Groups` distintos.

El protocolo DAP4 facilita identificar cada variable definida dentro de un `Group` con un [Fully Qualyfying Name](https://opendap.github.io/dap4-specification/DAP4.html#_fully_qualified_names) único. En este caso, el FQN de cada variable, similar a navegar por un sistema de archivos, es:

| VarName | FQN_VarName |
| :- | -: |
| `delta_time` | `/gt3r/heights/delta_time` |
| `lat_ph` | `/gt3r/heights/lat_ph` |
| `lon_ph` | `/gt3r/heights/lon_ph` |


Con este conocimiento, queremos solicitar solo estas variables del _DMR_ del servidor. La especificación DAP4 para Constraint Expressions es:


$$
\text{Data_url + ?dap4.ce=<FQN_VarName1>;<FQN_VarName2>;<FQN_VarName3> }
$$

Probémoslo ahora:


In [ ]:
data_urlCE = data_url+'?dap4.ce=/gt3r/heights/delta_time;/gt3r/heights/lat_ph;/gt3r/heights/lon_ph'

In [ ]:
%%time
ds = open_url(data_urlCE, protocol='dap4')

### Esto es un orden de magnitud más rápido que antes.


Como `Groups` NO forman parte del modelo de datos DAP2, no podemos ilustrar la CE en DAP2 con el dataset `ATLAS03`. En su lugar, ahora miramos el dataset `COADS`, que tiene arreglos 2D.



## Subconjunto espacial
Continuando con la demostración de `CEs` en el modelo de datos DAP4, ahora queremos solicitar solo un subconjunto de las variables. Primero observamos el dataset completo.


In [ ]:
%%time
data_url = 'http://test.opendap.org:8080/opendap/ghrsst/20210102090000-JPL-L4_GHRSST-SSTfnd-MUR-GLOB-v02.0-fv04.1.h5'

In [ ]:
%%time
ds = open_url(data_url, protocol='dap4')

In [ ]:
ds.tree()

Igual que antes en el modelo de datos DAP4, tenemos:

| VarName | FQN_VarName |
| :- | -: |
| `time` | `/time` |
| `lat` | `/lat` |
| `lon` | `/lon` |
| `mask` | `/mask` |
| `sea_ice_fraction` | `/sea_ice_fraction` |
| `dt_1km_data` | `/dt_1km_data` |
| `analysed_sst` | `/analysed_sst` |
| `analysis_error` | `/analysis_error` |
| `sst_anomaly` | `/sst_anomaly` |


In [ ]:
ds['analysed_sst'].shape

In [ ]:
ds['analysed_sst'].attributes

In [ ]:
print('Dimensions of variable:', ds['analysed_sst'].dimensions)

In [ ]:
ds.dimensions

Vemos que la variable tiene tres dimensiones, y estas coinciden con las del dataset completo.

### Subconjunto espacial

`Todavía no hemos descargado/recuperado ningún dato`. Solo PyDAP ha solicitado hasta ahora el _DMR_. Considera el escenario donde solo queremos inspeccionar un subconjunto espacial de los datos, definido por los `rangos de índices`, también llamados `hyperslabs`.

Supongamos que queremos recuperar solo los primeros 100 puntos de `lat` y los últimos 300 puntos de `lon`, de todas las variables del dataset. En DAP4 hay dos opciones para lograr esto con la URL.



1. `Enfoque tradicional`. Define el hyperslab para cada variable que solicites. El [Data Request Form](http://test.opendap.org:8080/opendap/ghrsst/20210102090000-JPL-L4_GHRSST-SSTfnd-MUR-GLOB-v02.0-fv04.1.h5.dmr) permite a las personas usuarias construir dichas URLs de forma interactiva (seleccionando casillas y modificando hyperslabs). Siguiendo este enfoque, obtienes la siguiente URL restringida:

```
data_url + ?dap4.ce=/time;/lat[0:1:100];/lon[35700:1:];/mask[0][0:1:100][35700:1:];/sea_ice_fraction[0][0:1:100][35700:1:];/dt_1km_data[0][0:1:100][35700:1:];/analysed_sst[0][0:1:100][35700:1:];/analysed_sst[0][0:1:100][35700:1:];/analysis_error[0][0:1:100][35700:1:];/sst_anomaly[0][0:1:100][35700:1:]
```

Probémoslo.


In [ ]:
%%time
nds = open_url(data_url+'?dap4.ce=/time[0];/lat[0:1:300];/lon[35699:1:35999];/mask[0][0:1:300][35699:1:35999];/sea_ice_fraction[0][0:1:300][35699:1:35999];/dt_1km_data[0][0:1:300][35699:1:35999];/analysed_sst[0][0:1:300][35699:1:35999];/analysis_error[0][0:1:300][35699:1:35999];/sst_anomaly[0][0:1:300][35699:1:35999]', protocol='dap4')

In [ ]:
nds

In [ ]:
nds['/analysed_sst'].shape

In [ ]:
nds['/lon'].shape

El subconjunto espacial descrito arriba funcionó bien. Es muy verboso y tienes que definir explícitamente el tamaño de todas las variables si quieres que el dataset subconjunto resultante sea autoconsistente. Por ejemplo, puede ser fácil definir el subconjunto espacial solo en una variable y no en el resto.


In [ ]:
%%time
nds = open_url(data_url+'?dap4.ce=/time;/lat;/lon;/mask;/sea_ice_fraction;/dt_1km_data;/analysed_sst[0][0:1:300][35699:1:35999]', protocol='dap4')

In [ ]:
nds['lat'].shape != nds['analysed_sst'].shape

In [ ]:
nds['lat'].shape, nds['analysed_sst'].shape

A continuación proporcionamos un enfoque alternativo que es menos propenso a errores y usa [Shared Dimensions](https://opendap.github.io/dap4-specification/DAP4.html#_subsetting_and_shared_dimensions).

2. `Shared Dimensions`: este enfoque alternativo se puede usar para definir el subconjunto espacial mediante las dimensiones. La sintaxis es:

```
data_url + ?dap4.ce=<FQN_Dim1>=[subset];<FQN_Dim2>=[subset];<FQN_Var1>;<FQN_Var3>;<FQN_Var3>;...<FQN_VarN>
```

Donde `<FQN_Var1>` puede ser igual a `<FQN_Dim1>`.

En la sintaxis anterior,

a) El subconjunto se define primero en las dimensiones mediante los signos `=`. Las dimensiones pueden ser globales (a nivel raíz) o estar dentro de un `Group`.

b) Luego la persona usuaria define las variables que PyDAP debe incluir en la solicitud. El subconjunto definido en el paso anterior se aplicará a la variable.



Usando el ejemplo, la URL mucho más simplificada se convierte en:


In [ ]:
%%time
nds = open_url(data_url+"?dap4.ce=/lat=[0:1:300];/lon=[35699:1:35999];/time;/lat;/lon;/mask;/sea_ice_fraction;/dt_1km_data;/analysed_sst;/analysis_error;/sst_anomaly", protocol='dap4')

In [ ]:
nds.tree()

In [ ]:
nds['/analysed_sst'].shape == nds['sst_anomaly'].shape